In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain_teddynote import logging

logging.langsmith("CH12-RAG")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH12-RAG


In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해주세요."
        ), # 시스템 프롬프트는 이렇고
        MessagesPlaceholder(variable_name="chat_history"), # 지금까지는 이렇게 질문과 대답을 했으니
        ("human", "#Question:\n{question}"), # 이번에는 이런 질문을 하겠다
    ]
)

llm=ChatOpenAI(model="gpt-6-luna")

chain=prompt | llm | StrOutputParser()

In [15]:
store = {}

def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store:
        store[session_ids]=ChatMessageHistory()
    return store[session_ids]

chain_with_history = RunnableWithMessageHistory(
    chain, # The base Runnable to be wrapped.
    get_session_history, # Function that returns a new BaseChatMessageHistory.
    input_messages_key="question", # Must be specified if the base runnable accepts a dict as input.
    history_messages_key="chat_history", # Must be specified if the base runnable accepts a dict as input and expects a separate key for historical messages.
)

In [16]:
chain_with_history.invoke(
    {"question": "나의 이름은 테디입니다."},
    config={"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


'알겠습니다, 테디님.'

In [17]:
chain_with_history.invoke(
    {"question": "내 이름이 뭐라고?"},
    config={"configurable": {"session_id":"abc123"}}
)

[대화 세션ID]: abc123


'테디님입니다.'

In [18]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
try:
    embeddings = OpenAIEmbeddings()

    # 실제 호출 테스트
    embeddings.embed_query("test")

    print("OpenAI 임베딩 사용")

except Exception as e:
    print(f"OpenAI 임베딩 사용 불가: {e}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "mps"},
        encode_kwargs={"normalize_embeddings": True},
    )

    print("HuggingFace 임베딩으로 대체")

OpenAI 임베딩 사용 불가: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 53379.32it/s]


HuggingFace 임베딩으로 대체


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

loader = PDFPlumberLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

embeddings = embeddings # 앞에서 미리 함

vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

